### 2 · Ingestion — Embed CMS3 Log Chunks into Pinecone

This notebook takes the processed CMS3 log chunks from Notebook 1 and upserts them into a Pinecone index.

### Pipeline
```
data/processed/cms3_log_chunks.json
  → Load JSON into LangChain Documents
  → Embed with OpenAI text-embedding-3-small
  → Upsert vectors into Pinecone
  → Reconnect to the index
  → Validate with sample debugging queries
```

The goal here is simple: take the retrieval-ready log chunks and make them searchable in the same managed vector store we plan to use in production.


In [1]:
import hashlib
import json
import os
from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec

# Resolve notebook-relative paths so the notebook works from Jupyter and scripted runs.
NOTEBOOK_DIR = Path.cwd()
load_dotenv(dotenv_path=NOTEBOOK_DIR.parent / ".env", override=True)

PROCESSED_DIR = NOTEBOOK_DIR.parent / "data/processed"
CHUNKS_PATH = PROCESSED_DIR / "cms3_log_chunks.json"

EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIMENSION = 1536
INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "move-mind-ai")
INDEX_HOST = os.getenv("PINECONE_INDEX_HOST")
NAMESPACE = os.getenv("PINECONE_NAMESPACE", "cms3-logs")
PINECONE_CLOUD = os.getenv("PINECONE_CLOUD", "aws")
PINECONE_REGION = os.getenv("PINECONE_REGION", "us-east-1")

required_env = ["OPENAI_API_KEY", "PINECONE_API_KEY"]
missing = [key for key in required_env if not os.getenv(key)]
if missing:
    raise ValueError(f"Missing required environment variables: {missing}")

print(f"Processed file: {CHUNKS_PATH}")
print(f"Pinecone index: {INDEX_NAME}")
print(f"Namespace: {NAMESPACE}")
print(f"Host override: {INDEX_HOST or '<not set>'}")

Processed file: /Users/sauravmajumdar/Developer/AI/move-mind-ai/data/processed/cms3_log_chunks.json
Pinecone index: move-mind-ai
Namespace: cms3-logs
Host override: https://move-mind-ai-g57210f.svc.aped-4627-b74a.pinecone.io


/Users/sauravmajumdar/Developer/AI/move-mind-ai/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1 — Load Processed Chunks

Notebook 1 exported the CMS3 chunks in the standard:
```json
{
  "page_content": "...",
  "metadata": {...}
}
```

So ingestion is mostly reconstruction: load the JSON and convert each row back into a LangChain `Document`.


In [2]:
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    raw_chunks = json.load(f)

# Rebuild LangChain Documents from the exported JSON structure.
documents = [
    Document(page_content=chunk["page_content"], metadata=chunk["metadata"])
    for chunk in raw_chunks
]

print(f"Loaded {len(documents)} documents from {CHUNKS_PATH.name}")
print("\nSample document metadata:")
pprint(documents[0].metadata)
print("\nSample content preview:")
print(documents[0].page_content[:800])

Loaded 87 documents from cms3_log_chunks.json

Sample document metadata:
{'chunk_type': 'execution_summary',
 'customer_id': '7093495',
 'event_count': 78,
 'execution_id': 'exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88',
 'first_step_order': 1,
 'group_key': 'cid:7093495 | '
              'exec:exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88',
 'journey_id': 'ccflownew',
 'last_step_order': 78,
 'route_count': 22,
 'source': 'cms3_logs',
 'status': 'success'}

Sample content preview:
group_key: cid:7093495 | exec:exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88
journey_id: ccflownew
customer_ids: ['7093495']
execution_ids: ['exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88']
event_count: 78
step_range: 1 -> 78
status_counts: {'success': 78}
top_actions: {'route_entered': 22, 'evaluate_ui_condition': 16, 'patch_context': 11, 'graphql_request': 9, 'navigate': 8, 'updateContext': 2, 'gql_get_available_agentsfrom_flex_query': 2, 'gql_create_lead_new': 2}
route_path: /ccflownew/ret

## Step 2 — Inspect the Chunk Mix

Before embedding, verify that the chunk distribution matches the preprocessing design.

We expect:
- many `event` chunks
- a smaller number of `execution_summary` chunks

This step is useful because ingestion bugs often come from malformed exports rather than embeddings themselves.


In [3]:
from collections import Counter

chunk_type_dist = Counter(doc.metadata.get("chunk_type") for doc in documents)
customer_dist = Counter(
    doc.metadata.get("customer_id")
    for doc in documents
    if doc.metadata.get("customer_id")
)
action_dist = Counter(
    doc.metadata.get("action") for doc in documents if doc.metadata.get("action")
)

print("Chunk types:", dict(chunk_type_dist))
print("Customers:", dict(customer_dist))
print("Top actions:")
for action, count in action_dist.most_common(10):
    print(f"  {action:<35} {count:>3}")

Chunk types: {'execution_summary': 2, 'event': 85}
Customers: {'7093495': 79}
Top actions:
  route_entered                        24
  evaluate_ui_condition                18
  patch_context                        12
  graphql_request                      10
  navigate                              8
  updateContext                         2
  gql_get_available_agentsfrom_flex_query   2
  gql_create_lead_new                   2
  gql_lsa_lead_on_answer_call           1
  gql_get_property_details_by_address   1


## Step 3 — Create Embeddings and Connect Pinecone

Now we initialize the embedding model and the Pinecone client.

Why this matters:
- `event` chunks stay searchable as exact evidence
- `execution_summary` chunks stay searchable as broader journey context
- Pinecone gives us the same managed retrieval backend we want outside local development


In [4]:
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

# Create the index on first run so the notebook is self-contained.
existing_indexes = {index_info["name"] for index_info in pc.list_indexes()}
if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBEDDING_DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION),
    )
    print(f"Created Pinecone index: {INDEX_NAME}")
else:
    print(f"Using existing Pinecone index: {INDEX_NAME}")

index = pc.Index(host=INDEX_HOST) if INDEX_HOST else pc.Index(INDEX_NAME)
vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace=NAMESPACE,
)

print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Target namespace: {NAMESPACE}")

Using existing Pinecone index: move-mind-ai
Embedding model: text-embedding-3-small
Target namespace: cms3-logs


## Step 4 — Upsert Chunks into Pinecone

Instead of saving a local FAISS index, we upsert each chunk into Pinecone.

We generate stable IDs so repeated notebook runs update the same vectors instead of creating duplicates.


In [5]:
def sanitize_metadata_value(value):
    if value is None:
        return None
    if hasattr(value, "item"):
        value = value.item()
    if isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, list):
        sanitized = [sanitize_metadata_value(item) for item in value]
        return [item for item in sanitized if item is not None]
    return str(value)


def sanitize_document_for_pinecone(doc: Document) -> Document:
    metadata = {
        key: sanitized
        for key, value in doc.metadata.items()
        if (sanitized := sanitize_metadata_value(value)) is not None
    }
    return Document(page_content=doc.page_content, metadata=metadata)


def build_doc_id(doc: Document) -> str:
    payload = json.dumps(
        {"page_content": doc.page_content, "metadata": doc.metadata},
        sort_keys=True,
        default=str,
    )
    return hashlib.sha1(payload.encode("utf-8")).hexdigest()


documents_for_upsert = [sanitize_document_for_pinecone(doc) for doc in documents]
doc_ids = [build_doc_id(doc) for doc in documents_for_upsert]
print(f"Upserting {len(documents_for_upsert)} chunks into Pinecone...")
upserted_ids = vectorstore.add_documents(documents=documents_for_upsert, ids=doc_ids)
print(f"Upsert complete. Returned {len(upserted_ids)} ids.")
print("Sample ids:", upserted_ids[:3])

Upserting 87 chunks into Pinecone...
Upsert complete. Returned 87 ids.
Sample ids: ['a921751972d5a333f77800b792fa9b5cd6613d90', '600ba27d67cd19c747c13d20a31ca03bef0b1581', 'c47d90b6701704877bb8fa850ca6dec4e9638b3a']


## Step 5 — Reconnect and Validate Retrieval

This step mirrors how the real app will use the store:
- connect to the managed Pinecone index
- create a retriever
- run sample debugging questions

If retrieval looks wrong here, the issue is likely in preprocessing, metadata design, or the index contents rather than LangGraph.


In [6]:
# Recreate the store object to mimic a fresh app process.
reloaded_store = PineconeVectorStore(
    index=pc.Index(host=INDEX_HOST) if INDEX_HOST else pc.Index(INDEX_NAME),
    embedding=embeddings,
    namespace=NAMESPACE,
)
retriever = reloaded_store.as_retriever(search_kwargs={"k": 5})

print(f"Connected to Pinecone index: {INDEX_NAME}")
print(f"Using namespace: {NAMESPACE}")

Connected to Pinecone index: move-mind-ai
Using namespace: cms3-logs


## Step 6 — Test Queries for the CMS3 Debugging Use Case

We validate the index with the kinds of questions users will eventually ask inside the Admin Tool.


In [7]:
def show_results(results, label="Results"):
    print(f"\n{label} ({len(results)} chunks)")
    print("-" * 100)
    for i, doc in enumerate(results, start=1):
        metadata = doc.metadata
        print(
            f"[{i}] type={metadata.get('chunk_type')} | "
            f"cid={metadata.get('customer_id')} | "
            f"exec={metadata.get('execution_id')} | "
            f"action={metadata.get('action')} | "
            f"page={metadata.get('page_path')}"
        )
        print(doc.page_content[:350])
        print()


TEST_QUERIES = [
    "For CID 7093495, what happened in this journey?",
    "Why did this customer end up on /ccflownew/quote/booking?",
    "Which UI condition evaluated to true on quote booking?",
    "What happened after move-scope for this customer?",
]

for query in TEST_QUERIES:
    print("=" * 100)
    print(f"Query: {query}")
    results = retriever.invoke(query)
    show_results(results, label="Retrieved chunks")

Query: For CID 7093495, what happened in this journey?

Retrieved chunks (5 chunks)
----------------------------------------------------------------------------------------------------
[1] type=event | cid=7093495 | exec=exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88 | action=route_entered | page=/ccflownew/isnj-journey
timestamp: 2026-04-02T06:51:49.308Z
journey_id: ccflownew
execution_id: exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88
customer_id: 7093495
step_order: 32
action: route_entered
event_type: navigation
status: success
level: info
page_path: /ccflownew/isnj-journey
source: router
message: Route entered
result_json: {"pathname": "/ccflownew/isnj-jou

[2] type=event | cid=7093495 | exec=exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88 | action=route_entered | page=/ccflownew/date
timestamp: 2026-04-02T06:50:51.925Z
journey_id: ccflownew
execution_id: exec-7093495-95fdafb2-9657-4cf0-aed9-31d98658fd88
customer_id: 7093495
step_order: 14
action: route_entered
event_type:

## Step 7 — What We Learned

This notebook now teaches the ingestion stage for the CMS3 debugging assistant:
- load processed log chunks
- rebuild `Document` objects
- embed them and upsert into Pinecone
- reconnect to the managed index
- validate retrieval on realistic debugging questions

At this point, the data is ready for Notebook 3, where we compare retrieval strategies specifically for log-debugging queries.


In [8]:
print("Notebook 2 is now aligned to Pinecone ingestion.")
print(f"Index: {INDEX_NAME}")
print(f"Namespace: {NAMESPACE}")

Notebook 2 is now aligned to Pinecone ingestion.
Index: move-mind-ai
Namespace: cms3-logs
